In [ ]:
from src.state import InputState, Context
from src.graph.workflow import build_graph
from IPython.display import Image, display

import os

os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
graph = build_graph()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
final_state = graph.invoke(
        input=InputState({
            "student_id": "student_01",
            "dpi": 300,
            "max_pages": 3,
            "max_regrade": 1,
            "rubric_file": "java_criteria.txt",
            }),
        context=Context(
            model="gpt-4o",
            temperature=0.0
            ),
		# config={"configurable": {"thread_id": "1"}}
        )

In [ ]:
print("\nFinal state:")
final_state.pop("messages")
for k, v in final_state.items():
	print(f"- {k}: {v}")

In [ ]:
from src.utils import io

stream = graph.stream(
        input=InputState({
            "student_id": "student_01",
            "dpi": 300,
            "max_pages": 3,
            "max_regrade": 1,
            "rubric_file": "java_criteria.txt",
            }),
        context=Context(
            model="gpt-4o",
            temperature=0.0
            ),
          # config={"configurable": {"thread_id": "1"}},
		stream_mode="updates",
          subgraphs=True,
		version="v2"
        )

for chunk in stream:
     if chunk["ns"]:
          node, values = list(chunk["data"].items())[0]
          print(f"\n- Subgraph[{node}] update\n  Messages:")
          for message in values.get("messages", []):
               if isinstance(message, tuple):
                    print(message)
               else:
                    message.content = io.truncate_text(message.content, limit=20)
                    message.pretty_print()
     else:
          print(f"\n* Root[{list(chunk['data'].keys())[0]}] update\n" + "-" * 100)